# 04 Main Ablation: Periodic Re-grounding

Run the main CS415 ablation on the locked 19-clip subset using `Grounding DINO + SAM2` with periodic re-grounding every 10 frames.

- Uses the same bootstrap path as D1 and the baseline: `bash setup_colab.sh --with-models`
- Compares directly against the official no-re-grounding baseline on the same subset
- Produces per-clip ablation outputs, `ablation_summary.json`, `ablation_review_table.csv`, and `ablation_delta_table.csv`


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/ChiCoTheLaAnh/ProjectFinalCS415.git'
REPO_DIR = '/content/ProjectFinalCS415'

%cd /content
!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"


In [ ]:
%cd /content/ProjectFinalCS415
!bash setup_colab.sh --with-models


In [ ]:
import shutil
import subprocess
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is not None:
    gpu_name = subprocess.run(
        [nvidia_smi, '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print('GPU:', gpu_name)

if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required for the re-grounding ablation run.')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/cv-final-project')
MANIFEST_PATH = Path('/content/ProjectFinalCS415/data/processed/subset_manifest.csv')
BASELINE_ROOT = DRIVE_ROOT / 'results' / 'quantitative' / 'baseline_groundingdino_sam2_no_regrounding'
ABLATION_ROOT = DRIVE_ROOT / 'results' / 'quantitative' / 'ablation_regrounding_every_10'
GROUNDING_CKPT = DRIVE_ROOT / 'checkpoints' / 'groundingdino_swint_ogc.pth'
SAM2_CKPT = DRIVE_ROOT / 'checkpoints' / 'sam2.1_hiera_small.pt'
DEVICE = 'cuda'
SUBSET_LIMIT = None

ABLATION_ROOT.mkdir(parents=True, exist_ok=True)
print('Manifest path:', MANIFEST_PATH)
print('Baseline root:', BASELINE_ROOT)
print('Ablation root:', ABLATION_ROOT)


In [ ]:
required_paths = [
    MANIFEST_PATH,
    BASELINE_ROOT / 'baseline_table.csv',
    GROUNDING_CKPT,
    SAM2_CKPT,
]
for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required path: {required_path}')
    print(required_path)


In [ ]:
import pandas as pd

manifest_df = pd.read_csv(MANIFEST_PATH)
manifest_df['selected'] = manifest_df['selected'].astype(str)
chosen = manifest_df[manifest_df['selected'] == '1'].copy()
display(chosen[['clip_id', 'primary_tag', 'notes']])
print('Selected clips:', len(chosen))
print(chosen['primary_tag'].value_counts())


In [ ]:
%cd /content/ProjectFinalCS415
import subprocess

cmd = [
    'python3', 'scripts/run_ablation.py',
    '--config', 'configs/base.yaml',
    '--manifest', str(MANIFEST_PATH),
    '--baseline_dir', str(BASELINE_ROOT),
    '--output_dir', str(ABLATION_ROOT),
    '--grounding_ckpt', str(GROUNDING_CKPT),
    '--sam2_ckpt', str(SAM2_CKPT),
    '--device', DEVICE,
]
if SUBSET_LIMIT is not None:
    cmd.extend(['--limit', str(SUBSET_LIMIT)])
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json

summary_path = ABLATION_ROOT / 'ablation_summary.json'
review_table_path = ABLATION_ROOT / 'ablation_review_table.csv'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing ablation summary: {summary_path}')
if not review_table_path.exists():
    raise FileNotFoundError(f'Missing ablation review table: {review_table_path}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
review_df = pd.read_csv(review_table_path)
display(review_df)


## Manual Review Step

Open `ablation_review_table.csv` in the ablation output root and review all rows using the same label space as the baseline.

Allowed `review_label` values:

- `good_tracking`
- `partial_tracking`
- `drift`
- `wrong_object`
- `no_detection`
- `fallback`

Each row must also have a one-line `review_note`.


In [ ]:
ablation_review_df = pd.read_csv(ABLATION_ROOT / 'ablation_review_table.csv')
display(ablation_review_df)
print('Review label counts:')
print(ablation_review_df['review_label'].fillna('').value_counts())


In [ ]:
%cd /content/ProjectFinalCS415
import subprocess

cmd = [
    'python3', 'scripts/run_ablation.py',
    '--manifest', str(MANIFEST_PATH),
    '--baseline_dir', str(BASELINE_ROOT),
    '--output_dir', str(ABLATION_ROOT),
    '--finalize_reviewed',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
delta_table_path = ABLATION_ROOT / 'ablation_delta_table.csv'
summary_path = ABLATION_ROOT / 'ablation_summary.json'
if not delta_table_path.exists():
    raise FileNotFoundError(f'Missing delta table: {delta_table_path}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
delta_df = pd.read_csv(delta_table_path)
display(delta_df)
print(json.dumps(summary, indent=2))
print('Delta counts:', summary.get('delta_counts', {}))
print('Review counts:', summary.get('review_counts', {}))
